# Daytime Radiative Cooling — Figures Only

This notebook creates figures from the already prepared file `processed/model_input_5min.csv`. It does **not** merge, resample, interpolate, smooth, normalise, or train a model.

Run `main_clean.ipynb` first if `model_input_5min.csv` does not yet exist. All figures made here are saved in `processed/figures_main_3/`.

## 1. Setup

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Works whether the notebook is opened from code/ or from the project root.
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'code' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'processed' / 'model_input_5min.csv'
FIG_DIR = PROJECT_ROOT / 'processed' / 'figures_main_3'
FIG_DIR.mkdir(parents=True, exist_ok=True)

GRID = '5min'
FEATURES = ['T_amb', 'RH', 'v', 'G', 'T_sky']
TARGET = 'Q_net'
PLOT_ORDER = [TARGET, 'G', 'T_amb', 'T_sky', 'RH', 'v']

LABELS = {
    'T_amb': ('Ambient temperature', r'$T_{amb}$ [$^\circ$C]', '#1f77b4'),
    'RH':    ('Relative humidity',   'RH [%]',                  '#2ca02c'),
    'v':     ('Wind speed',          r'$v$ [$m\,s^{-1}$]',     '#9467bd'),
    'G':     ('Solar irradiance',    r'$G$ [$W\,m^{-2}$]',     '#ff7f0e'),
    'T_sky': ('Sky temperature',     r'$T_{sky}$ [$^\circ$C]', '#d62728'),
    'Q_net': ('Net heat flux',       r'$Q_{net}$ [$W\,m^{-2}$]', '#111111'),
}

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'font.size': 11,
    'axes.titlesize': 13,
})

## 2. Load the prepared model table

The table should already contain one row per valid five-minute slot and the six physical variables required for the project.

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f'{DATA_PATH} was not found. Run main_clean.ipynb before this figures-only notebook.'
    )

data = pd.read_csv(DATA_PATH)
data['timestamp'] = pd.to_datetime(data['timestamp'], errors='raise')
data = data[['timestamp'] + FEATURES + [TARGET]].sort_values('timestamp').reset_index(drop=True)

if data['timestamp'].duplicated().any():
    raise ValueError('The prepared table has duplicate timestamps.')
if data[FEATURES + [TARGET]].isna().any().any():
    raise ValueError('The prepared table contains missing values. Fix preprocessing before plotting.')

print(f'Loaded {len(data):,} rows from {DATA_PATH.name}')
print(f'Time span: {data.timestamp.min()}  →  {data.timestamp.max()}')
data.head()

## 3. Build a plotting view that preserves real data gaps

The original table deliberately omits long instrument outages. Reindexing only for plotting inserts `NaN` at these missing five-minute timestamps, so Matplotlib leaves a visible break rather than drawing a misleading line across an outage.

In [ ]:
full_time_index = pd.date_range(data.timestamp.min(), data.timestamp.max(), freq=GRID)
plot_data = data.set_index('timestamp').reindex(full_time_index)
plot_data.index.name = 'timestamp'

missing_slots = plot_data[TARGET].isna().sum()
print(f'Plotting slots: {len(plot_data):,}')
print(f'Visible gap slots: {missing_slots:,} (these are not interpolated in this notebook)')

## 4. Data coverage figure

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.5))

present = data.timestamp.iloc[::5]
for row, col in enumerate(PLOT_ORDER):
    ax.plot(present, np.full(len(present), row), '|', color=LABELS[col][2], markersize=10)

ax.set_yticks(range(len(PLOT_ORDER)))
ax.set_yticklabels(PLOT_ORDER)
ax.invert_yaxis()
ax.set_xlabel('Time')
ax.set_title('Coverage of the prepared five-minute model table')
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
fig.tight_layout()
fig.savefig(FIG_DIR / '00_model_data_coverage.png')
plt.show()

## 5. All six model variables

In [ ]:
fig, axes = plt.subplots(len(PLOT_ORDER), 1, figsize=(15, 16), sharex=True)

for ax, col in zip(axes, PLOT_ORDER):
    title, ylabel, colour = LABELS[col]
    ax.plot(plot_data.index, plot_data[col], color=colour, linewidth=0.75)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{title} ({col})', loc='left', fontsize=12)
    if col == TARGET:
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)

axes[-1].set_xlabel('Time')
axes[-1].xaxis.set_major_locator(mdates.AutoDateLocator())
axes[-1].xaxis.set_major_formatter(mdates.ConciseDateFormatter(mdates.AutoDateLocator()))
fig.suptitle('Prepared model variables — five-minute data', y=0.997, fontsize=15)
fig.tight_layout()
fig.savefig(FIG_DIR / '01_all_model_variables.png')
plt.show()

## 6. Individual publication-size figures

In [ ]:
for col in PLOT_ORDER:
    title, ylabel, colour = LABELS[col]
    fig, ax = plt.subplots(figsize=(15, 5))
    ax.plot(plot_data.index, plot_data[col], color=colour, linewidth=0.8)
    if col == TARGET:
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.5)

    locator = mdates.AutoDateLocator(minticks=5, maxticks=12)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
    ax.set_title(f'{title} vs Time', fontsize=15)
    ax.set_xlabel('Time')
    ax.set_ylabel(ylabel)
    fig.tight_layout()

    for extension in ('png', 'pdf'):
        fig.savefig(FIG_DIR / f'{col}_vs_time.{extension}')
    plt.show()

print(f'Saved {len(PLOT_ORDER)} individual figures as PNG and PDF in {FIG_DIR}')

## 7. Mean daily cycle: net heat flux and solar irradiance

In [ ]:
hourly = data.set_index('timestamp')[[TARGET, 'G']].groupby(lambda timestamp: timestamp.hour).mean()

fig, ax1 = plt.subplots(figsize=(11, 5))
ax2 = ax1.twinx()

line_qnet, = ax1.plot(hourly.index, hourly[TARGET], color=LABELS[TARGET][2], marker='o',
                      markersize=4, linewidth=2, label=r'$Q_{net}$')
line_g, = ax2.plot(hourly.index, hourly['G'], color=LABELS['G'][2], marker='s',
                   markersize=4, linewidth=2, label=r'$G$')

ax1.axhline(0, color='black', linewidth=0.8, alpha=0.4)
ax1.set_xlabel('Hour of day')
ax1.set_ylabel(LABELS[TARGET][1])
ax2.set_ylabel(LABELS['G'][1], color=LABELS['G'][2])
ax2.tick_params(axis='y', colors=LABELS['G'][2])
ax2.grid(False)
ax1.set_xticks(range(0, 24, 2))
ax1.set_title('Mean daily cycle — net heat flux against solar irradiance')
ax1.legend(handles=[line_qnet, line_g], loc='upper left')
fig.tight_layout()
fig.savefig(FIG_DIR / '02_mean_daily_cycle.png')
plt.show()

## 8. Correlation heatmap

This is descriptive only. Correlation does not prove causality or determine model performance.

In [ ]:
corr = data[FEATURES + [TARGET]].corr()

fig, ax = plt.subplots(figsize=(7.5, 6.2))
image = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns)

for row in range(len(corr)):
    for column in range(len(corr)):
        value = corr.iloc[row, column]
        ax.text(column, row, f'{value:.2f}', ha='center', va='center',
                fontsize=10, color='white' if abs(value) > 0.6 else 'black')

ax.set_title('Pearson correlation of model variables')
ax.grid(False)
fig.colorbar(image, ax=ax, shrink=0.82, label='Correlation')
fig.tight_layout()
fig.savefig(FIG_DIR / '03_correlation_heatmap.png')
plt.show()

## Output

This notebook creates only visual outputs. The source data is never overwritten.

In [ ]:
created = sorted(path.name for path in FIG_DIR.iterdir() if path.is_file())
print(f'Figure folder: {FIG_DIR}')
print(f'Files present: {len(created)}')
for filename in created:
    print(' -', filename)